[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/rattleback/rattleback.ipynb)

# The Rattleback

This example computes the motion of a rattleback, a boat-shaped top that spins willingly one way and not the other: spun the wrong way it rocks, stops and spins back, and released tipped from rest it rocks into spin. Its underside is an ellipsoid rolling on a table without slipping, and its mass sits along axes turned a few degrees against the underside's; that small misalignment couples rocking to spin and gives one sense the preference.

The body's points and directions are antivectors, and its underside a support map from planes to directions: the table's plane through the centre, read in the body, goes to the direction of the lowest point, `-shape(normal) / height`, with the height the pairing `normal & shape(normal)` under its square root. Rolling holds that point still, so a spin about the centre is carried to the contact by the translation there, the product of two points `contact * origin.inverse()`, and `rolling` is a map from spins to rates. Its adjugate pulls forques back onto spins; the contact's reaction does no work under any rolling motion and drops out, so one solve of `pullback(inertia(rolling))` gives the angular acceleration.

In the vector notation of rigid-body mechanics rolling without slipping reads as $\mathbf v = \boldsymbol\omega \times \mathbf r$, with $\mathbf r$ the lever from the contact to the centre, and the contact force drops out of the balance of moments taken about the contact point.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from itertools import islice

import numpy as np

from numga import stack
from numga.algebras import PGA3D as ga
from examples import instantiate
from examples.animation import save_figure, show_animation
from examples.mechanics.rattleback import render

In [ ]:
core = instantiate("examples.mechanics.rattleback.core", ga)
mv = core.mv

Scalar = ga.gatype.scalar()
# Planes through the origin; points and directions are antivectors, intrinsic.
Plane = ga.gatype.vector().derive.nondegenerate
Point = ga.gatype.antivector()
Direction = Point.derive.degenerate
Spin = ga.gatype.bivector().derive.nondegenerate
Rate = ga.gatype.bivector()
Motor = ga.gatype.rotor()
Forque = ga.gatype.antibivector()
AngularMomentum = Forque.derive.degenerate

infinity = mv.w                                                 # [] Plane
origin = infinity.dual()                                        # [] Point
up = mv.z.dual()                                                # [] Direction

## 1. Shape and mass

The underside is an ellipsoid, given twice: as the stretch of the unit sphere onto it, for drawing, and as its support map, for rolling. Six equal masses sit inside the shell with the second moment of a solid ellipsoid, their axes turned nine degrees against the shell's; the orange masses and their axes show the misalignment, and the shell itself has negligible mass.

In [ ]:
# The shell's semi-axes, the body's mass, gravity, the turn of the masses' axes, and the share of the
# shell's height its flat deck sits at.
semi_axes = np.array([5, 17 / 16, 1])
mass = 4
gravity_strength = 9.8
mass_angle = np.deg2rad(9)
deck_fraction = 0.8

In [ ]:
# The planes across the shell's axes, and the axes' directions.
mirrors = stack((mv.x, mv.y, mv.z))                             # [3] Plane
directions = mirrors.dual()                                     # [3] Direction
outline = (directions * (mirrors & Direction) * semi_axes).sum(axis=0)       # Direction <- Direction
shape = (directions * (Plane & directions) * semi_axes**2).sum(axis=0)       # Direction <- Plane

# The masses along axes turned within the shell, half a span from the centre each way.
mass_axes = (mv.xy * (-mass_angle / 2)).exp() >> directions    # [3] Direction
offsets = ((mass_axes * semi_axes)[:, None] * np.array([-1, 1]) * np.sqrt(3 / 5)).reshape(-1)   # [masses] Direction
mass_points = origin + offsets                                  # [masses] Point

# Each mass joined with its own motion under a rate is its momentum; the inertia is their mean, times the mass.
inertia = (mass_points & mass_points.commutator(Rate)).mean(axis=0) * mass   # Forque <- Rate
gravity = -up * gravity_strength                                # [] Direction
save_figure(render.draw_setup(outline, mass_points, deck_fraction), "rattleback_mass_axes");

## 2. Rolling inertia

The table holds the lowest point while the body turns about it, and as the body rocks that point wanders over the shell, so the allowed motions change with the pose. The rolling map's adjugate pulls forques back to spins while keeping their work, so in `pullback(inertia(rolling))` the one constraint decides both the motion and its inertia, and the reaction at the contact never appears.

In [ ]:
def motion(motor: Motor, spin: Spin) -> tuple[Rate, Spin]:
    """The rolling body's rate and angular acceleration, the contact's reaction eliminated."""
    # The table's plane read in the body, the lowest point's direction from the centre, and the height.
    normal = (motor << up).dual_inverse()                       # Plane
    height = (normal & shape(normal)).square_root()             # Scalar
    contact = -shape(normal) / height                           # Direction

    # A spin about the centre, carried to the contact by the translation there, turns the body about it.
    rolling = (contact * origin.inverse() / 2).exp() >> Spin    # Rate <- Spin
    rate = rolling(spin)                                        # Rate

    # As the body turns, the table's plane moves in it, and the lowest point with it.
    normal_rate = spin.commutator(normal)                       # Plane
    height_rate = (normal & shape(normal_rate)) / height        # Scalar
    contact_rate = -shape(normal_rate) / height - contact * height_rate / height   # Direction
    rolling_change = (contact_rate * origin.inverse()).commutator(spin)   # Rate
    force = (origin & (motor << gravity)) * mass                # Forque

    # The contact's reaction does no work under any rolling motion: the pullback removes it.
    pullback = rolling.adjugate()                               # AngularMomentum <- Forque
    effective = pullback(inertia(rolling))                      # AngularMomentum <- Spin
    acceleration = effective.solve(pullback(
        force - inertia(rate).commutator(rate) - inertia(rolling_change)
    ))                                                          # Spin
    return rate, acceleration

## 3. Rocking into spin

Released from a fifteen-degree tilt with no spin, the body pitches and rolls before settling into rotation, its raised centre of mass supplying the energy, and the flat deck shows the changing lean. This cell takes about twenty seconds, simulating and drawing.

In [ ]:
# The two launches: tipped from rest, and spinning the unfavourable way with a little pitching. The time
# step and the steps, the steps between frames drawn, the time the rocking animation runs, and the
# duration of a frame.
initial_tilts = np.deg2rad([15, 0])
initial_spins = np.array([0, -1.5])
initial_pitch_rates = np.array([0, 0.1])
dt = 0.04
n_steps = 250
frame_stride = 2
rocking_duration = 8
n_rocking_steps = round(rocking_duration / dt)
duration_ms = round(dt * frame_stride * 1000)

In [ ]:
spin = mv.xy * initial_spins - mv.xz * initial_pitch_rates      # [cases] Spin
tilt = (-mv.xz * initial_tilts / 2).exp()                        # [cases] Motor
# Each launch lifted by its centre's height, so that the shell rests on the table.
normal = (tilt << up).dual_inverse()                            # [cases] Plane
height = (normal & shape(normal)).square_root()                 # [cases] Scalar
motor = (up * origin.inverse() * height / 2).exp() * tilt       # [cases] Motor

# Placements and body-frame spins stepped together on the motion group.
states = islice(core.evolve(motor, spin, dt, motion), n_steps + 1)
placements, spins = (stack(list(history)) for history in zip(*states))   # [steps + 1, cases] each
show_animation(render.frames(placements[:n_rocking_steps + 1:frame_stride, 0], outline, deck_fraction),
               "rattleback_rocking", duration_ms)

## 4. Spin reversal

Launched clockwise with a small pitching disturbance, the body loses its spin to rocking, then turns the other way.

In [ ]:
show_animation(render.frames(placements[::frame_stride, 1], outline, deck_fraction),
               "rattleback_reversal", duration_ms)

In [ ]:
# checks
# The tipped body spins up from rest, the spinning one reverses, and the energy is kept.
energies = core.energy(placements, spins, shape, inertia, mass, gravity, up)   # [steps + 1, cases] Scalar
spun = (-(placements >> spins).scalar_product(mv.xy)).to_array()   # [steps + 1, cases]: the spin about the vertical
assert abs(spun[0, 0]) < 1e-11 and spun[-1, 0] > 1.4
assert spun[0, 1] < -1.4 and spun[-1, 1] > 1.4
np.testing.assert_allclose((energies / energies[0]).to_array(), 1, rtol=1e-2)

The rolling ellipsoid follows the full mechanical model in Arioli and Koch's [*Some Reversing Orbits for a Rattleback Model*](https://link.springer.com/article/10.1007/s00332-022-09797-7), with its nearly uniform, skewed inertia realized by six masses. Yoshida, Tokieda and Morrison's [*Rattleback: a model of how geometric singularity induces dynamic chirality*](https://arxiv.org/abs/1609.09223) studies the exchange between spin and rocking through a reduced dynamical system.